In [1]:
import os
from dotenv import load_dotenv; load_dotenv()  
from langchain_google_genai import ChatGoogleGenerativeAI


api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY environment variable not set")

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./../../public/nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

107


In [3]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FO

{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': './../../public/nke-10k-2023.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

516


In [5]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text:v1.5")
if embeddings is None:
    raise ValueError("Failed to initialize OllamaEmbeddings")
else: print("OllamaEmbeddings initialized successfully")

OllamaEmbeddings initialized successfully


In [6]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db", 
)

In [7]:
ids = vector_store.add_documents(documents=all_splits)

In [15]:
results = vector_store.similarity_search("How many distribution centers does Nike have in the US?", k=3)

print(results[0])

page_content='direct to consumer operations sell products through the following number of retail stores in the United States:
U.S. RETAIL STORES NUMBER
NIKE Brand factory stores 213 
NIKE Brand in-line stores (including employee-only stores) 74 
Converse stores (including factory stores) 82 
TOTAL 369 
In the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for further information.
2023 FORM 10-K 2' metadata={'moddate': '2023-07-20T16:22:08-04:00', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'title': '0000320187-23-000039', 'source': './../../public/nke-10k-2023.pdf', 'creator': 'EDGAR Filing HTML Converter', 'page': 4, 'total_pages': 107, 'start_index': 3125, 'creationdate': '2023-07-20T16:22:00-04:00', 'keywords': '0000320187-23-000039; ; 10-K', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'page_label': '5'}


```python
# Async Query can be used as well
results = await vector_store.asimilarity_search("When was Nike incorporated?")
```

In [ ]:
results = vector_store.similarity_search_with_relevance_scores("What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.7731049487790884

page_content='Table of Contents
FISCAL 2023 NIKE BRAND REVENUE HIGHLIGHTSThe following tables present NIKE Brand revenues disaggregated by reportable operating segment, distribution channel and major product line:
FISCAL 2023 COMPARED TO FISCAL 2022
• NIKE, Inc. Revenues were $51.2 billion in fiscal 2023, which increased 10% and 16% compared to fiscal 2022 on a reported and currency-neutral basis, respectively.
The increase was due to higher revenues in North America, Europe, Middle East & Africa ("EMEA"), APLA and Greater China, which contributed approximately 7, 6,
2 and 1 percentage points to NIKE, Inc. Revenues, respectively.
• NIKE Brand revenues, which represented over 90% of NIKE, Inc. Revenues, increased 10% and 16% on a reported and currency-neutral basis, respectively. This
increase was primarily due to higher revenues in Men's, the Jordan Brand, Women's and Kids' which grew 17%, 35%,11% and 10%, respectively, on a wholesale
equivalent basis.' metad

#### ⚠️ This score is a distance, NOT relevance

- Lower = more similar
- Higher = less similar

In [16]:
# Batch Retrieval

from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)


retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

[[Document(id='e33d7f48-363d-4c5e-aada-35342bcbbbe4', metadata={'creationdate': '2023-07-20T16:22:00-04:00', 'source': './../../public/nke-10k-2023.pdf', 'start_index': 3125, 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'title': '0000320187-23-000039', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'total_pages': 107, 'page': 4, 'creator': 'EDGAR Filing HTML Converter', 'moddate': '2023-07-20T16:22:08-04:00', 'keywords': '0000320187-23-000039; ; 10-K', 'page_label': '5'}, page_content='direct to consumer operations sell products through the following number of retail stores in the United States:\nU.S. RETAIL STORES NUMBER\nNIKE Brand factory stores 213 \nNIKE Brand in-line stores (including employee-only stores) 74 \nConverse stores (including factory stores) 82 \nTOTAL 369 \nIn the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for fur

# OR

In [17]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

[[Document(id='e33d7f48-363d-4c5e-aada-35342bcbbbe4', metadata={'page': 4, 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'start_index': 3125, 'creationdate': '2023-07-20T16:22:00-04:00', 'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'keywords': '0000320187-23-000039; ; 10-K', 'source': './../../public/nke-10k-2023.pdf', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'creator': 'EDGAR Filing HTML Converter', 'page_label': '5', 'title': '0000320187-23-000039', 'total_pages': 107, 'moddate': '2023-07-20T16:22:08-04:00'}, page_content='direct to consumer operations sell products through the following number of retail stores in the United States:\nU.S. RETAIL STORES NUMBER\nNIKE Brand factory stores 213 \nNIKE Brand in-line stores (including employee-only stores) 74 \nConverse stores (including factory stores) 82 \nTOTAL 369 \nIn the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for fur

In [1]:
from langchain_community.document_loaders import WebBaseLoader

loader_one = WebBaseLoader("https://docs.langchain.com/oss/python/integrations/document_loaders/web_base")
loader_multiple_pages = WebBaseLoader(
    ["https://www.example.com/", "https://google.com"]
)

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
docs_one = loader_one.load()
print(f"Loaded {len(docs_one)} document(s) from single URL")

Loaded 1 document(s) from single URL


In [6]:
print(docs_one); print(docs_one[0].metadata)

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/integrations/document_loaders/web_base', 'title': 'WebBaseLoader integration - Docs by LangChain', 'description': 'Integrate with the WebBaseLoader document loader using LangChain Python.', 'language': 'en'}, page_content='WebBaseLoader integration - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationWebBaseLoader integrationDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonLangChain integrationsAll providersPopular ProvidersOpenAIAnthropicGoogleAWSHugging FaceMicrosoftOllamaGroqIntegrations by componentChat modelsTools and toolkitsMiddlewareRetrieversText splittersEmbedding modelsVector storesDocument loadersKey-value storesOn this pageOverviewIntegration detailsLoader featuresSetupCredentialsInstallationInitializationInitialization with multiple pagesLoadLoad multiple urls concurrentlyLoading a xml file, 

You can speed up the scraping process by scraping and parsing multiple urls concurrently.

In [1]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.example.com/")

USER_AGENT environment variable not set, consider setting it to identify your requests.
